# Healthcare Analytics: Comprehensive Model Evaluation
This notebook computes the detailed metrics, confusion matrices, and ROC-AUC curves for the trained models.


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, precision_recall_curve, average_precision_score
)

sns.set_theme(style='whitegrid')


## 1. Load Data & Models


In [ ]:
X_test = pd.read_csv(os.path.join('..', 'data', 'temp', 'X_test_scaled.csv'))
y_test = pd.read_csv(os.path.join('..', 'data', 'temp', 'y_test.csv')).values.ravel()

models = {
    'Logistic Regression': joblib.load(os.path.join('..', 'models', 'logistic_regression_model.pkl')),
    'Decision Tree': joblib.load(os.path.join('..', 'models', 'decision_tree_model.pkl')),
    'Random Forest': joblib.load(os.path.join('..', 'models', 'random_forest_model.pkl')),
    'XGBoost': joblib.load(os.path.join('..', 'models', 'xgboost_model.pkl'))
}


## 2. Generate Classification Reports


In [ ]:
for name, model in models.items():
    preds = model.predict(X_test)
    print('='*50)
    print(f'Model: {name}')
    print('='*50)
    print(classification_report(y_test, preds))
    print('\n')


## 3. Confusion Matrix Visualization


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (name, model) in enumerate(models.items()):
    preds = model.predict(X_test)
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Stroke', 'Stroke'])
    disp.plot(ax=axes[i], cmap='Blues', colorbar=False)
    axes[i].set_title(f'{name} Confusion Matrix')
    axes[i].grid(False)

plt.tight_layout()
plt.show()


## 4. ROC-AUC Curves Comparison


In [ ]:
plt.figure(figsize=(10, 8))

for name, model in models.items():
    probs = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})', lw=2)

plt.plot([0, 1], [0, 1], color='navy', linestyle='--', label='Random Guessing')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()


## 5. Precision-Recall Curves Comparison
Especially useful for imbalanced datasets.


In [ ]:
plt.figure(figsize=(10, 8))

for name, model in models.items():
    probs = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, probs)
    ap = average_precision_score(y_test, probs)
    plt.plot(recall, precision, label=f'{name} (AP = {ap:.3f})', lw=2)

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='lower left')
plt.show()
